# 05 · Recursión final (patrón iterativo) y `foldLeft`

> **En el examen:** casi siempre dentro del **Ejercicio 3** o **4**, justo después de pedir
> la versión recursiva normal: *"implementa la función mediante **recursión final**"* y
> *"implementa la función mediante **`foldLeft`**"* (≈ 1-2 puntos cada una).

Requisito: haber leído el archivo 04 (recursión normal y `foldRight`).

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

## 1. ¿Qué es la recursión final y para qué sirve?

En la recursión normal (archivo 04), después de la llamada recursiva todavía queda trabajo:

```scala
case head :: tail => head + suma(tail)     // primero suma(tail), DESPUÉS sumar head
```

El ordenador tiene que "acordarse" de cada `head` pendiente, y con listas muy largas la
memoria (la *pila*) se desborda.

Una función es **recursiva final** (*tail recursive*) si la llamada recursiva es **lo
último** que hace: no queda nada pendiente después. Para conseguirlo, se lleva la solución
**acumulada** en un parámetro.

Es exactamente un **bucle** escrito sin variables mutables. Compara:

In [ ]:
// Imperativo (con var y while)
def sumaI(l: List[Int]): Int =
    var out = 0                // 1. inicializo la solución
    var aux = l
    while aux != Nil do
        out = out + aux.head   // 2. actualizo con el elemento actual
        aux = aux.tail
    out                        // 3. devuelvo

// Recursión final: lo mismo
def sumaTR(l: List[Int]): Int =
    def step(out: Int, aux: List[Int]): Int =
        aux match
            case Nil => out                              // 3. se acabó: devuelvo
            case head :: tail => step(out + head, tail)  // 2. actualizo y sigo
    step(0, l)                                           // 1. inicializo

---

## 2. La plantilla del patrón iterativo (viene en el preámbulo de algunos exámenes)

In [ ]:
def iterative[A, B](l: List[A]): B =

    def step(out: B, aux: List[A]): B =
        aux match
            case Nil => out                              // fin: la solución parcial es la final
            case head :: tail =>
                step(???  /* actualizar out con head */, tail)

    step(??? /* valor inicial */, l)

Tres preguntas que tienes que responder:

1. **¿Con qué valor empieza `out`?** (el resultado para la lista vacía)
2. **¿Cómo actualizo `out` al ver un elemento `head`?**
3. **¿Hace falta retocar el resultado al final?** (por ejemplo `.reverse`, o quedarte con un
   componente de una tupla)

`step` es una **función auxiliar definida dentro** de la principal (archivo 01, sección 2).
Así puede usar los parámetros de la función principal (por ejemplo `l2` en `intersect`).

Opcional: escribir `@annotation.tailrec` encima de `def step` hace que Scala **compruebe**
que de verdad es recursiva final (si no lo es, da error).

---

## 3. Ejemplo guiado: `fromBits` (25-26 Extraordinaria)

`fromBits(List(1, 0, 1, 1)) == 11` (binario 1011). Pista del enunciado: recorre de izquierda
a derecha con un acumulador que empieza en 0 y se actualiza con `out * 2 + bit`.

1. inicial: `0`
2. actualizar: `out * 2 + head`
3. al final: nada

In [ ]:
def fromBits(l: List[Int]): Int =
    def step(out: Int, aux: List[Int]): Int =
        aux match
            case Nil => out
            case head :: tail => step(out * 2 + head, tail)
    step(0, l)

Traza para `List(1, 0, 1, 1)`: out = 0 → 1 → 2 → 5 → 11.

---

## 4. `foldLeft`: el bucle ya escrito

In [ ]:
def foldLeft[B](initial: B)(update: (B, A) => B): B
//              ^^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^
//              valor       cómo actualizar (out, elemento)
//              inicial

Qué hace, para `List(1, 2, 3)`:

```
List(1, 2, 3).foldLeft(z)(f)  ==  f(f(f(z, 1), 2), 3)
```

Empieza **por la izquierda**: combina `z` con el primero, el resultado con el segundo, etc.

Su implementación es la plantilla de la sección 2:

In [ ]:
def foldLeft[A, B](l: List[A])(initial: B)(update: (B, A) => B): B =
    l match
        case Nil => initial
        case head :: tail => foldLeft(tail)(update(initial, head))(update)

### Receta: de recursión final a `foldLeft`

| En la versión con `step` | En `foldLeft` |
|---|---|
| el valor inicial `step(INICIAL, l)` | primer argumento de `foldLeft` |
| lo que pasas como nuevo `out` en `step(NUEVO, tail)` | el cuerpo de `(out, head) => NUEVO` |
| el retoque final (`.reverse`, `._1`…) | se pone **después** del `foldLeft` |

In [ ]:
def fromBits(l: List[Int]): Int =
    l.foldLeft(0)((out, bit) => out * 2 + bit)

// o con placeholders: el primer _ es out y el segundo es bit
def fromBits2(l: List[Int]): Int =
    l.foldLeft(0)(_ * 2 + _)

### ⚠️ El orden de los argumentos es el contrario que en `foldRight`

- `foldRight`: la función recibe `(elemento, solucionDelResto)`
- `foldLeft`: la función recibe `(acumulado, elemento)`

Un truco para recordarlo: el acumulado está **del lado** del nombre (Left → acumulado a la
izquierda; Right → solución a la derecha).

---

## 5. Construir listas: ¿por qué hace falta `.reverse`?

Al recorrer de izquierda a derecha y **añadir delante** con `::`, el primer elemento que
ves acaba al **final**:

```
out = Nil
ve 1 → 1 :: Nil          = List(1)
ve 2 → 2 :: List(1)      = List(2, 1)
ve 3 → 3 :: List(2, 1)   = List(3, 2, 1)     ← al revés
```

Solución estándar: **añade delante (barato) y da la vuelta al final** con `.reverse`.
(Añadir al final con `out :+ x` también funciona, pero es lento; los exámenes aceptan la
versión con `reverse`, que es la que usa el profesor.)

### `intersect` (24-25 Móstoles)

In [ ]:
// recursión final
def intersect[A](l1: List[A], l2: List[A]): List[A] =
    def step(out: List[A], aux: List[A]): List[A] =
        aux match
            case Nil => out
            case head :: tail if l2.contains(head) => step(head :: out, tail)
            case _ :: tail => step(out, tail)
    step(Nil, l1).reverse

// foldLeft
def intersectFL[A](l1: List[A], l2: List[A]): List[A] =
    l1.foldLeft(Nil: List[A]):
        case (out, e) if l2.contains(e) => e :: out
        case (out, _) => out
    .reverse

Igual que en `foldRight`: si necesitas pattern matching, los `case` reciben **el par**
`(out, elemento)`. Y el `.reverse` en una línea aparte, al nivel del `foldLeft`, se aplica al
resultado del `foldLeft`.

### `pair` (23-24 Extraordinaria)

In [ ]:
// pair(1, List(2,3,4)) == List((1,2), (1,3), (1,4))
def pair[A, B](e: A, l: List[B]): List[(A, B)] =
    def step(out: List[(A, B)], aux: List[B]): List[(A, B)] =
        aux match
            case Nil => out
            case h :: t => step((e, h) :: out, t)
    step(Nil, l).reverse

def pairFL[A, B](e: A, l: List[B]): List[(A, B)] =
    l.foldLeft(Nil: List[(A, B)])((acc, e2) => (e, e2) :: acc).reverse

---

## 6. Llevar información extra: acumuladores que son tuplas

Muchas veces, para decidir qué hacer con un elemento, necesitas saber algo más que la
solución parcial: **la posición**, **el elemento anterior**, **un contador**… Solución: el
acumulador es una **tupla**, y al final te quedas con lo que te interesa.

### Con posición: `slice` (21-22 Vicálvaro)

`slice(List(1,2,3,4))(1, 3) == List(2, 3, 4)` (posiciones de la 1 a la 3).

In [ ]:
def slice[A](list: List[A])(from: Int, to: Int): List[A] =
    list.foldLeft((Nil: List[A], 0)):           // (resultado al revés, posición actual)
        case ((out, idx), e) if from <= idx && idx <= to => (e :: out, idx + 1)
        case ((out, idx), _) => (out, idx + 1)
    ._1.reverse                                  // me quedo con el resultado y lo doy la vuelta

### Con el anterior (`Option`): `isSorted` (23-24 Vicálvaro)

In [ ]:
def isSorted[A](l: List[A])(lessEq: (A, A) => Boolean): Boolean =
    l.foldLeft((true, None: Option[A])):          // (¿ordenada hasta ahora?, anterior)
        case ((_, None), e) => (true, Some(e))     // primer elemento: no hay anterior
        case ((ok, Some(last)), e) => (ok && lessEq(last, e), Some(e))
    ._1

Versión con recursión final que mira dos elementos a la vez (también válida, y más corta):

In [ ]:
@annotation.tailrec
def isSortedTR[A](l: List[A])(lessEq: (A, A) => Boolean): Boolean =
    l match
        case a1 :: a2 :: tail if lessEq(a1, a2) => isSortedTR(a2 :: tail)(lessEq)
        case _ :: _ :: _ => false
        case _ => true            // 0 o 1 elementos

### Con un flag (`Boolean`): `odds` (19-20 Prueba)

Elementos en posiciones impares (1ª, 3ª, 5ª…):

In [ ]:
def oddsFL[A](list: List[A]): List[A] =
    list.foldLeft((List(): List[A], true)):       // (resultado, ¿meto el siguiente?)
        case ((out, true), e) => (e :: out, false)
        case ((out, false), _) => (out, true)
    ._1.reverse

### Con post-procesado: números romanos (24-25 Extraordinaria, 4 puntos)

`decode("MCMXXIV") == 1924`. Idea: llevar `(anterior, suma)`. Al ver un símbolo nuevo, el
**anterior** se suma o se resta según si es menor que el nuevo. Al acabar, queda sumar el
último anterior.

In [ ]:
val Sym: Char => Int =
    case 'I' => 1
    case 'V' => 5
    case 'X' => 10
    case 'L' => 50
    case 'C' => 100
    case 'D' => 500
    case 'M' => 1000

// recursión final
def decode(s: String): Int =
    def step(out: (Option[Int], Int), aux: List[Char]): (Option[Int], Int) =
        (out, aux) match
            case (out, Nil) => out
            case ((None, res), head :: tail) =>
                step((Some(Sym(head)), res), tail)
            case ((Some(prev), res), head :: tail) if prev < Sym(head) =>
                step((Some(Sym(head)), res - prev), tail)
            case ((Some(prev), res), head :: tail) =>
                step((Some(Sym(head)), res + prev), tail)

    step((None, 0), s.toList) match           // post-procesado
        case (None, e)      => e
        case (Some(e1), e2) => e1 + e2

// foldLeft
def decodeFL(s: String): Int =
    s.toList.foldLeft((None: Option[Int], 0)):
        case ((None, res), head) => (Some(Sym(head)), res)
        case ((Some(prev), res), head) if prev < Sym(head) => (Some(Sym(head)), res - prev)
        case ((Some(prev), res), head) => (Some(Sym(head)), res + prev)
    match
        case (None, e)      => e
        case (Some(e1), e2) => e1 + e2

(`s.toList` convierte el `String` en `List[Char]`. El `match` en la última parte se aplica
al resultado del `foldLeft`.)

### `grouped` (24-25 Vicálvaro): trozos de tamaño fijo

`grouped(List(1,2,3,4,5,6,7), 3) == List(List(1,2,3), List(4,5,6), List(7))`

In [ ]:
// recursión final, usando splitAt
def grouped[A](l: List[A], size: Int): List[List[A]] =
    def step(out: List[List[A]], aux: List[A]): List[List[A]] =
        aux match
            case Nil => out
            case _ =>
                val (chunk, rest) = aux.splitAt(size)   // separo los primeros size
                step(chunk :: out, rest)
    step(Nil, l).reverse

// foldLeft: el trozo actual está en la cabeza del acumulador, al revés
def groupedFL[A](l: List[A], size: Int): List[List[A]] =
    l.foldLeft(Nil: List[List[A]]):
        case (chunk :: rest, e) if chunk.size < size => (e :: chunk) :: rest   // cabe en el trozo actual
        case (out, e) => List(e) :: out                                         // empiezo trozo nuevo
    .reverse          // los trozos estaban al revés
    .map(_.reverse)   // y cada trozo también

### `split` con recursión final y `foldLeft` (25-26 Vicálvaro Bis)

> La solución del archivo del examen Bis repite la versión no final; esta es la que pide el
> enunciado.

In [ ]:
def splitTR[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    def step(out: (List[A], List[B]), aux: List[Either[A, B]]): (List[A], List[B]) =
        aux match
            case Nil => out
            case Left(a) :: tail  => step((a :: out._1, out._2), tail)
            case Right(b) :: tail => step((out._1, b :: out._2), tail)
    val (as, bs) = step((Nil, Nil), l)
    (as.reverse, bs.reverse)

def splitFL[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    val (as, bs) = l.foldLeft((List[A](), List[B]())):
        case ((xs, ys), Left(a))  => (a :: xs, ys)
        case ((xs, ys), Right(b)) => (xs, b :: ys)
    (as.reverse, bs.reverse)

### De imperativo a `foldLeft`: `sumDivisors` (22-23 Extraordinaria)

Te dan esto:

In [ ]:
def sumDivisorsF(n: Int): Int =
    var out = 0
    for (i <- 1 to n/2)
        if (n % i == 0) out += i
    out

Traducción directa (`var out = 0` → valor inicial; el cuerpo del bucle → la función):

In [ ]:
def sumDivisors(n: Int): Int =
    (1 to n/2).toList.foldLeft(0)((out, i) => if n % i == 0 then out + i else out)

### `foldLeft` con una función ya hecha: construir un árbol (19-20 Diciembre)

Si ya tienes `add(tree, i)` que inserta en un árbol de búsqueda (archivo 07), construir el
árbol a partir de una lista es un `foldLeft` donde la función de actualización es
**directamente** `add`:

In [ ]:
def toTree(list: List[Int]): Tree[Int] =
    list.foldLeft(void[Int])(add)       // add: (Tree[Int], Int) => Tree[Int]

---

## 7. ¿`foldLeft` o `foldRight`?

| Situación | Más natural |
|---|---|
| construir una lista en el mismo orden | `foldRight` (no necesita `reverse`) |
| un cálculo que va de izquierda a derecha (binario, romanos, "el anterior") | `foldLeft` |
| sumar, contar, máximo… | cualquiera |
| listas enormes (no desbordar la pila) | `foldLeft` / recursión final |

En el examen **te dicen cuál usar**. Úsala aunque la otra te parezca más fácil.

---

## 8. Errores típicos

- **La llamada recursiva no es lo último.** `head :: step(out, tail)` NO es recursión final
  (queda pendiente el `::`). Tiene que ser `step(algo, tail)` sin nada alrededor.
- **Olvidar el `.reverse`** al construir listas.
- **Invertir el orden de los parámetros** de la función de `foldLeft`: es `(acumulado, elemento)`.
- **No poner el tipo del valor inicial** (`Nil: List[A]`, `None: Option[Int]`).
- **Olvidar el post-procesado** cuando el acumulador es una tupla (`._1`, `match` final).

---

## 9. Ejercicios para practicar

### De exámenes reales (pide recursión final **y** `foldLeft`)

| Función | Examen |
|---|---|
| `fromBits` | 25-26 Extraordinaria |
| `split` (Either) | 25-26 Vicálvaro Bis |
| `decode` (números romanos) | 24-25 Extraordinaria |
| `intersect` | 24-25 Móstoles |
| `grouped` | 24-25 Vicálvaro |
| `pair` | 23-24 Extraordinaria |
| `isSorted` | 23-24 Vicálvaro |
| `sumDivisors` | 22-23 Extraordinaria |
| `flatten` de `Struct` (con `foldLeft`) | 22-23 Vicálvaro Especial |
| `distinct` | 21-22 Extraordinaria |
| `lift` (con `foldLeft`) | 21-22 Móstoles |
| `slice` | 21-22 Vicálvaro |
| `split` (pares y Either) | 20-21 Extraordinaria |
| `toTree`, `insertAt` | 19-20 Diciembre |
| `findIndex` | 19-20 Julio |
| `odds` | 19-20 Prueba |

### Ejercicios nuevos (recursión final y `foldLeft`)

1. `maximo(l: List[Int]): Option[Int]`
2. `reverse[A](l: List[A]): List[A]` (sin usar `.reverse`)
3. `indexOf[A](l: List[A], x: A): Option[Int]` — posición de la primera aparición
4. `runningSum(l: List[Int]): List[Int]` — `List(1,2,3)` → `List(1,3,6)`

<details>
<summary>Soluciones con foldLeft</summary>

```scala
def maximo(l: List[Int]): Option[Int] =
    l.foldLeft(None: Option[Int]):
        case (None, e) => Some(e)
        case (Some(m), e) => Some(if e > m then e else m)

def reverse[A](l: List[A]): List[A] =
    l.foldLeft(Nil: List[A])((out, e) => e :: out)

def indexOf[A](l: List[A], x: A): Option[Int] =
    l.foldLeft((0, None: Option[Int])):
        case ((i, None), e) if e == x => (i + 1, Some(i))
        case ((i, found), _) => (i + 1, found)
    ._2

def runningSum(l: List[Int]): List[Int] =
    l.foldLeft((0, Nil: List[Int])):
        case ((acc, out), e) => (acc + e, (acc + e) :: out)
    ._2.reverse
```

</details>

Siguiente: **06-funciones-orden-superior.ipynb**.